# Geodesics and Orbital Dynamics around Black Holes

Using EinsteinPy to simulate particle trajectories and geodesics around black holes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from einsteinpy.geodesic import Geodesic
from einsteinpy.plotting import StaticGeodesicPlotter
from astropy import units as u
from astropy import constants as const

print("EinsteinPy Geodesics - Loaded")

In [ ]:
# Notebook metadata
__author__ = "Astro Science Research"
__date__ = "2026-10-28"
__version__ = "1.0.0"
__kernel__ = "python3"
import sys
print(f"Python {sys.version}")
print(f"Notebook: {__file__ if '__file__' in dir() else 'interactive'}")


## 1. Schwarzschild Geodesics

First, let's simulate geodesics around a Schwarzschild (non-rotating) black hole.

In [ ]:
# Define Schwarzschild geodesic (EinsteinPy 0.4: metric name + metric_params)
pos = [10.0, np.pi/2, 0.0]  # Initial position (r, theta, phi)
mom = [0.0, 0.0, 1.5]  # Initial momentum

geod = Geodesic(
    "Schwarzschild", (),  # metric params: M = 1 in geometrized units
    position=pos,
    momentum=mom,
    steps=1000,
    delta=0.5,
    return_cartesian=True,
    suppress_warnings=True,
)

# Integrate
geod.calculate_trajectory()

# Plot
sgpl = StaticGeodesicPlotter()
sgpl.plot(geod)
sgpl.show()

## 2. Circular Orbits around Schwarzschild Black Hole

Calculate the radius of the innermost stable circular orbit (ISCO):

In [ ]:
def isco_radius_schwarzschild(mass):
    """
    ISCO radius for a Schwarzschild black hole.
    
    R_isco = 6 GM/c^2 = 3 R_s   (the correct value; a common slip is 6 R_s = 12 GM/c^2).
    """
    from astropy import constants as const
    rs = 2 * const.G * mass / const.c**2
    r_isco = 3 * rs  # = 6 GM/c^2
    return r_isco.to(u.km)

# ISCO for different masses
test_masses = [10 * u.Msun, 36 * u.Msun, 4.1e6 * u.Msun]

print("ISCO Radii:")
print("-" * 50)
for mass in test_masses:
    r_isco = isco_radius_schwarzschild(mass)
    rs = 2 * const.G * mass / const.c**2
    print(f"Mass: {mass:.2e} -> Rs: {rs.to(u.km):.2f}, ISCO: {r_isco:.2f}")

## 3. Kerr Black Hole Geodesics

Now let's simulate geodesics around a rotating (Kerr) black hole.

In [ ]:
# Kerr geodesic with spin parameter a (dimensionless, 0 = Schwarzschild, 1 = extremal)
a = 0.9

# Initial conditions for a prograde orbit
pos = [10.0, np.pi/2, 0.0]
mom = [0.0, 0.0, 1.8]  # Higher angular momentum for Kerr

geod_kerr = Geodesic(
    "Kerr", (a,),  # metric params: spin a
    position=pos,
    momentum=mom,
    steps=1000,
    delta=0.5,
    return_cartesian=True,
    suppress_warnings=True,
)

# Integrate
geod_kerr.calculate_trajectory()

# Plot
sgpl_kerr = StaticGeodesicPlotter()
sgpl_kerr.plot(geod_kerr)
sgpl_kerr.show()

## 4. Comparison: Schwarzschild vs Kerr ISCO

The ISCO depends on the spin parameter a:

In [ ]:
def isco_radius_kerr(mass, a, retrograde=False):
    """
    Exact Bardeen-Press-Teukolsky ISCO for a Kerr black hole (BPT 1972).
    
    r_ISCO / M = 3 + Z2 ∓ sqrt((3 - Z1)(3 + Z1 + 2 Z2)),
    Z1 = 1 + (1 - a^2)^(1/3) [(1+a)^(1/3) + (1-a)^(1/3)],
    Z2 = sqrt(3 a^2 + Z1^2)
    
    Upper sign (∓ = -) is prograde; retrograde uses +.
    Sanity: a=0 -> 6M; a=1 prograde -> 1M, retrograde -> 9M.
    (A straight-line chord 6-5a errs by up to ~36% near a=0.93; do not use it.)
    """
    from astropy import constants as const
    rg = const.G * mass / const.c**2  # gravitational radius GM/c^2
    z1 = 1 + (1 - a**2)**(1/3) * ((1 + a)**(1/3) + (1 - a)**(1/3))
    z2 = np.sqrt(3 * a**2 + z1**2)
    sign = +1 if retrograde else -1
    r_isco = rg * (3 + z2 + sign * np.sqrt((3 - z1) * (3 + z1 + 2 * z2)))
    return r_isco.to(u.km)

# Compare ISCO for different spins (GW150914 mass)
mass = 36 * u.Msun
spins = [0.0, 0.5, 0.9, 0.99]

print(f"ISCO for {mass} Black Hole:")
print("-" * 60)
for a in spins:
    r_pro = isco_radius_kerr(mass, a)
    r_ret = isco_radius_kerr(mass, a, retrograde=True)
    rg = const.G * mass / const.c**2
    print(f"Spin a={a:.2f} -> r_g: {rg.to(u.km):.2f}, ISCO prograde: {r_pro:.2f}, retrograde: {r_ret:.2f}")

## 5. Orbital Period

Calculate the orbital period at a given radius:

In [ ]:
def orbital_period(mass, radius):
    """
    Calculate orbital period at a given radius (Newtonian approximation).
    
    T = 2π * sqrt(r^3 / GM)
    """
    from astropy import constants as const
    T = 2 * np.pi * np.sqrt(radius**3 / (const.G * mass))
    return T.to(u.s)

# Orbital periods at different radii for Sgr A*
sgrA_mass = 4.1e6 * u.Msun
sgrA_rs = 2 * const.G * sgrA_mass / const.c**2

radii = [5*sgrA_rs, 10*sgrA_rs, 50*sgrA_rs, 100*sgrA_rs]

print(f"Orbital Periods near Sgr A*:")
print("-" * 70)
for r in radii:
    T = orbital_period(sgrA_mass, r)
    print(f"Radius: {r.to(u.km):.2e} -> Period: {T.to(u.hour):.2f}")

## Summary

This notebook covered:
1. Schwarzschild geodesics
2. ISCO calculation
3. Kerr black hole geodesics
4. Spin effects on ISCO
5. Orbital periods

Next: Gravitational wave modeling.